# 02 - Baseline CNN fra scratch

I denne notebooken bygger vi vår første CNN-modell fra bunnen av ved hjelp av Keras. Vi bruker `Conv2D`, `MaxPooling2D` og `Dense` lag for 
å løse et multi-label klassifiseringsproblem på røntgenbilder.

Vi fokuserer på de 5 klinisk viktigste sykdommene som Stanford valgte for 
CheXpert-konkurransen (Irvin et al., 2019):
- Atelectasis
- Cardiomegaly
- Consolidation
- Edema
- Pleural Effusion

Disse ble valgt basert på klinisk viktighet og prevalens i datasettet, og brukes 
av alle modeller på [CheXpert leaderboardet](https://stanfordmlgroup.github.io/competitions/chexpert/).

**Referanse:** Irvin, J. et al. (2019). CheXpert: A Large Chest Radiograph Dataset 
with Uncertainty Labels and Expert Comparison. *AAAI*. https://arxiv.org/abs/1901.07031

## 1. Importer biblioteker
Vi bruker Keras (med TensorFlow som backend) for å bygge CNN-modellen, 
slik vi har lært i DAT255. Keras gir oss tilgang til lag som `Conv2D`, 
`MaxPooling2D` og `Dense` som er kjernen i convolutional neural networks.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import tensorflow as tf
import keras
from keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.20.0
Keras: 3.13.2


## 2. Last inn data og definer labels

Vi fokuserer på de 5 klinisk viktigste sykdommene fra CheXpert-konkurransen.
Vi må også ta stilling til hvordan vi håndterer uncertain labels (-1).
Strategiene som CheXpert-paperet beskriver er:
- U-Zeros: mapper uncertain → 0 (negativ)
- U-Ones: mapper uncertain → 1 (positiv)
- U-Ignore: ignorerer uncertain under trening

Vi bruker U-Ones for Atelectasis og Edema, og U-Zeros for resten, 
i tråd med Stanford sin baseline